In [ ]:
import os
import dotenv
from google import genai


In [ ]:
dotenv.load_dotenv()  # Load environment variables from .env file
os.environ['GOOGLE_API_KEY']=os.getenv('GOOGLE_API_KEY')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import RetrievalQA
from langchain_community.vectorstores import MongoDBAtlasVectorSearch
from pymongo import MongoClient


In [ ]:
#testing the embedding model
client = genai.Client()

result = client.models.embed_content(
        model="gemini-embedding-001",
        contents="What is the meaning of life?"
)

print(result.embeddings)

[ContentEmbedding(
  values=[
    -0.022374554,
    -0.004560777,
    0.013309286,
    -0.0545072,
    -0.02090443,
    <... 3067 more items ...>,
  ]
)]


In [ ]:

from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

uri = os.getenv('MONGO_URI')

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [ ]:
db = client["rag_db"]
collection = db["documents"]

In [144]:
# Drop the collection to start fresh
collection.drop()
print("Collection dropped.")

Collection dropped.


In [145]:

# Step 1: Load Document
loader = PyPDFLoader(r"C:\Users\prasa\Downloads\Resume\Manikandan_P_Resume.pdf")
documents = loader.load()


In [138]:
import os
pdf_path = r"C:\Users\prasa\Downloads\Resume\Manikandan_P_Resume.pdf"
if os.path.exists(pdf_path):
    print(f"File exists, size: {os.path.getsize(pdf_path)} bytes")
    print("First few pages content:")
    print(documents[0].page_content[:500] if documents else "No documents loaded")
else:
    print("File does not exist")

File exists, size: 78061 bytes
First few pages content:
Manikandan P
Backend Developer & Machine Learning Engineer
+91 97903 24050 | prasadpmanikandan@gmail.com | LinkedIn | GitHub | Chennai, India
Profile Summary
Backend Developer with machine learning experience and skilled in Java and Spring Boot for building scalable
APIs and microservices. Experienced in developing ML models and integrating them with backend systems for
real-time predictions.
Skills
• Languages: Java, Python, SQL
• Frontend Technologies: HTML, CSS, JS, React.js
• Backend Technol


In [139]:
for i, doc in enumerate(documents):
    if "Tata" in doc.page_content:
        print(f"Found 'Tata' in page {i}: {doc.page_content[:200]}...")
        break
else:
    print("Tata not found in documents")

Found 'Tata' in page 0: Manikandan P
Backend Developer & Machine Learning Engineer
+91 97903 24050 | prasadpmanikandan@gmail.com | LinkedIn | GitHub | Chennai, India
Profile Summary
Backend Developer with machine learning ex...


In [146]:

# Step 2: Split Documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)
split_docs = text_splitter.split_documents(documents)

# print(docs[5].page_content)

In [ ]:
print("Number of split docs:", len(split_docs))
for i in range(min(3, len(split_docs))):    
    print(f"Doc {i}: {split_docs[i].page_content[:200]}...")

Number of split docs: 19
Doc 0: Manikandan P
Backend Developer & Machine Learning Engineer
+91 97903 24050 | prasadpmanikandan@gmail.com | LinkedIn | GitHub | Chennai, India
Profile Summary...
Doc 1: Profile Summary
Backend Developer with machine learning experience and skilled in Java and Spring Boot for building scalable...
Doc 2: APIs and microservices. Experienced in developing ML models and integrating them with backend systems for
real-time predictions.
Skills
• Languages: Java, Python, SQL...


In [147]:
# Step 3: Gemini Embeddings
embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)


In [168]:
test_embedding = embedding.embed_query("t")
print(f"Embedding dimensions: {len(test_embedding)}")

Embedding dimensions: 3072


In [165]:
# Step 4: MongoDB Vector Store
vectorstore = MongoDBAtlasVectorSearch.from_documents(
    split_docs,
    embedding,
    collection=collection,
    index_name="my_vector_index"
)

In [143]:
collection.create_search_index(
    {
        "name": "my_vector_index_3072",
        "definition": {
            "mappings": {
                "fields": {
                    "embedding": {
                        "type": "knnVector",
                        "dimensions": 3072,
                        "similarity": "cosine"
                    }
                }
            }
        }
    }
)

'my_vector_index_3072'

In [169]:
# try:
#     collection.drop_search_index("my_vector_index")
#     print("Dropped existing index")
# except Exception as e:
#     print(f"Error dropping index: {e}")

collection.create_search_index(
    {
        "name": "my_vector_index",
        "definition": {
            "mappings": {
                "fields": {
                    "embedding": {
                        "type": "knnVector",
                        "dimensions": 3072,
                        "similarity": "cosine"
                    }
                }
            }
        }
    }
)

'my_vector_index'

In [170]:
# Step 5: Retriever
retriever = vectorstore.as_retriever()

In [176]:
docs = retriever.invoke("backend developer")

print(docs)

[Document(metadata={'_id': ObjectId('69c9418a08c1a8d621f0b411'), 'embedding': [0.00044698716, 0.0026144814, 0.019365642, -0.08338241, -0.01934605, 0.017645605, 0.0054419124, -0.001867494, -0.011234887, 0.006323173, -0.017359594, -0.0039819703, -0.0036644668, 0.022283448, 0.13774575, 0.004630731, -0.00811551, -0.012051464, 0.0056817, -0.0031004676, -0.0003553479, 0.014824354, -0.01670456, -0.011584852, -0.0074033123, -0.018299611, 0.027123291, 0.016562065, 0.038544632, -0.001151486, -0.014842126, 0.021629883, 0.007544656, 0.024841107, -0.008945266, 0.0089067165, 0.014873718, -0.013852835, -0.0012331521, 0.013466958, 0.00790979, 0.014921926, -0.011710057, -0.012306463, -0.0036278304, 0.010953693, -0.0048710788, 0.0042879456, 0.006509769, 0.02087014, -0.0026221587, 0.0062008277, -0.03154633, -0.23077442, 0.002490547, -0.0028525793, -0.016236616, -0.009915478, 0.024421467, -0.009056382, -0.021003986, 0.024339795, -0.023102185, 0.01544457, -0.0013107827, -0.0019827771, 0.029178403, 0.000818

In [172]:
# Step 6: Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0.1
)


In [173]:
# Step 7: RAG Chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

In [ ]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain


template=ChatPromptTemplate.from_template("""You are a helpful assistant, answer the question only with the
                 provided context and with your knowledge
     
                <context>
                {context}
                </context>
                Question:{input}
     """)
chain = create_stuff_documents_chain(llm, template)

# Retrieval Chain
retrieval_chain = create_retrieval_chain(
    retriever,
    chain
)
print(retrieval_chain)



In [ ]:
# Step 7: Ask Questions
while True:
    query = input("\nAsk a question: ")

    if query.lower() == "exit":
        break

    result = qa.run(query)

    print("\nAnswer:", result)